# 02 - Explorative Datenanalyse: Forex-Daten (EODHD API)

**Ziel:** Forex-Kursdaten von EODHD laden, erkunden und erste Qualitaetspruefung durchfuehren.

**Waehrungspaare:** EUR/USD, EUR/CHF, GBP/USD

**Datenquelle:** EODHD API (https://eodhd.com)

**API-Dokumentation:** https://eodhd.com/financial-apis/api-for-historical-data-and-volumes

---

## 1. Setup und Imports

In [ ]:
# Bibliotheken importieren
import requests              # HTTP-Anfragen an die API
import pandas as pd          # Datenverarbeitung
import numpy as np           # Numerische Berechnungen
import matplotlib.pyplot as plt  # Visualisierung
import seaborn as sns        # Erweiterte Visualisierung
import os                    # Dateipfade
from dotenv import load_dotenv  # API-Key aus .env laden

# Darstellung konfigurieren
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

print('Setup erfolgreich!')

## 2. API-Key laden

Der API-Key wird aus der `.env` Datei geladen (wird nicht ins Git hochgeladen).

In [ ]:
# .env Datei laden (liegt im Hauptordner, ein Verzeichnis hoeher)
load_dotenv(dotenv_path='../.env')

# API-Key auslesen
api_key = os.getenv('EODHD_API_KEY')

# Pruefen ob Key vorhanden ist (ohne ihn anzuzeigen)
if api_key and api_key != 'dein_api_key_hier':
    print(f'API-Key geladen (beginnt mit: {api_key[:4]}...)')
else:
    print('FEHLER: Kein API-Key gefunden! Bitte .env Datei pruefen.')

## 3. Daten laden von EODHD API

Die EODHD API liefert Forex-Daten im Format:

`https://eodhd.com/api/eod/EURUSD.FOREX?from=2024-01-01&to=2025-12-31&period=d&api_token=KEY&fmt=json`

Die Antwort ist ein JSON-Array mit: date, open, high, low, close, adjusted_close, volume

In [ ]:
# Konfiguration: Waehrungspaare und Zeitraum (gleich wie Yahoo Finance!)
CURRENCY_PAIRS = {
    'EURUSD.FOREX': 'EUR/USD',
    'EURCHF.FOREX': 'EUR/CHF',
    'GBPUSD.FOREX': 'GBP/USD',
}

START_DATE = '2024-01-01'
END_DATE = '2025-12-31'

print(f'Zeitraum: {START_DATE} bis {END_DATE}')
print(f'Waehrungspaare: {list(CURRENCY_PAIRS.values())}')

In [ ]:
# Daten von der EODHD API laden
forex_data = {}

for eodhd_symbol, pair_name in CURRENCY_PAIRS.items():
    print(f'Lade {pair_name} ({eodhd_symbol})...')
    
    # API-URL und Parameter zusammenbauen
    url = f'https://eodhd.com/api/eod/{eodhd_symbol}'
    params = {
        'from': START_DATE,
        'to': END_DATE,
        'period': 'd',
        'api_token': api_key,
        'fmt': 'json'
    }
    
    # HTTP-GET Request ausfuehren
    response = requests.get(url, params=params)
    
    # Status pruefen
    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        forex_data[pair_name] = df
        print(f'  -> {len(df)} Zeilen geladen')
    else:
        print(f'  -> FEHLER: HTTP {response.status_code}')

print('\nAlle Daten geladen!')

## 4. Erste Datenuebersicht

In [ ]:
# Uebersicht fuer jedes Waehrungspaar
for pair_name, df in forex_data.items():
    print(f'\n{"=" * 50}')
    print(f'{pair_name}')
    print(f'{"=" * 50}')
    print(f'Shape (Zeilen x Spalten): {df.shape}')
    print(f'Zeitraum: {df.index.min()} bis {df.index.max()}')
    print(f'Spalten: {list(df.columns)}')
    print(f'Datentypen:\n{df.dtypes}')
    print(f'\nErste 3 Zeilen:')
    display(df.head(3))

## 5. Datenqualitaetspruefung

In [ ]:
# Fehlende Werte pruefen
print('FEHLENDE WERTE PRO WAEHRUNGSPAAR')
print('=' * 50)

for pair_name, df in forex_data.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    print(f'\n{pair_name}:')
    if missing.sum() == 0:
        print('  Keine fehlenden Werte!')
    else:
        for col in df.columns:
            if missing[col] > 0:
                print(f'  {col}: {missing[col]} fehlend ({missing_pct[col]}%)')

In [ ]:
# Duplikate pruefen
print('DUPLIKATE PRO WAEHRUNGSPAAR')
print('=' * 50)

for pair_name, df in forex_data.items():
    dupes = df.index.duplicated().sum()
    print(f'{pair_name}: {dupes} Duplikate')

In [ ]:
# Deskriptive Statistik
for pair_name, df in forex_data.items():
    print(f'\n{"=" * 50}')
    print(f'Deskriptive Statistik: {pair_name}')
    print(f'{"=" * 50}')
    display(df.describe())

## 6. Visualisierung

### 6.1 Kursverlauf (Close-Preis)

In [ ]:
# Kursverlauf plotten
fig, axes = plt.subplots(len(forex_data), 1, figsize=(14, 4 * len(forex_data)))

if len(forex_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, forex_data.items()):
    ax.plot(df.index, df['close'], label=f'{pair_name} (EODHD)', linewidth=1)
    ax.set_title(f'Kursverlauf {pair_name} (EODHD)', fontsize=14)
    ax.set_xlabel('Datum')
    ax.set_ylabel('Kurs')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Verteilung der taeglichen Renditen

In [ ]:
# Taegliche Renditen
fig, axes = plt.subplots(1, len(forex_data), figsize=(6 * len(forex_data), 5))

if len(forex_data) == 1:
    axes = [axes]

for ax, (pair_name, df) in zip(axes, forex_data.items()):
    returns = df['close'].pct_change().dropna()
    ax.hist(returns, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Taegliche Renditen: {pair_name}', fontsize=12)
    ax.set_xlabel('Rendite (%)')
    ax.set_ylabel('Haeufigkeit')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 6.3 Fehlende Tage identifizieren

In [ ]:
# Alle fehlenden Tage anzeigen
for pair_name, df in forex_data.items():
    print(f'\n{"=" * 50}')
    print(f'{pair_name}')
    print(f'{"=" * 50}')
    
    all_days = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    missing_days = all_days.difference(df.index)
    missing_weekends = [d for d in missing_days if d.weekday() >= 5]
    missing_weekdays = [d for d in missing_days if d.weekday() < 5]
    
    print(f'Gesamte fehlende Tage:     {len(missing_days)}')
    print(f'  davon Wochenenden:       {len(missing_weekends)}')
    print(f'  davon Wochentage:        {len(missing_weekdays)}')
    
    if len(missing_weekdays) > 0:
        print(f'\n  Fehlende Wochentage (Feiertage etc.):')
        for d in missing_weekdays:
            day_name = ['Mo', 'Di', 'Mi', 'Do', 'Fr'][d.weekday()]
            print(f'    {d.strftime("%Y-%m-%d")} ({day_name})')

## 7. Rohdaten speichern

In [ ]:
# Rohdaten als CSV speichern
OUTPUT_DIR = '../data/raw/forex/eodhd'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for pair_name, df in forex_data.items():
    safe_name = pair_name.replace('/', '_')
    filename = f'{safe_name}_{START_DATE}_to_{END_DATE}.csv'
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath)
    print(f'Gespeichert: {filepath} ({len(df)} Zeilen)')

print('\nAlle Rohdaten gespeichert!')

## 8. Zusammenfassung

### Erkenntnisse aus der EDA:
- **Datenumfang:** (hier Ergebnisse eintragen nach Ausfuehrung)
- **Fehlende Werte:** (hier Ergebnisse eintragen)
- **Duplikate:** (hier Ergebnisse eintragen)
- **Auffaelligkeiten:** (hier Ergebnisse eintragen)

### Naechste Schritte:
1. Datenqualitaet zwischen Yahoo und EODHD vergleichen
2. Daten bereinigen und harmonisieren
3. Nachrichten laden (EODHD News API + Webscraping)